# v14b — Representation Extraction (checkpointed)

Reads `datasets/unified/injection_corpus.csv` from **v14a** and writes hidden states to
`representation_cache/unified/`.

**This is the only notebook that needs the model.** Everything after it is cache-only.

## Design
- **Sharded and resumable.** Rows are processed in fixed blocks of `SHARD`; each finished
  shard is written immediately as `shard_XXXX.npz`. Re-running skips completed shards, so
  a crash, an OOM or closing the lid costs at most one shard.
- **Row alignment is guaranteed.** Each shard stores the corpus row indices it covers.
  v14c reconstructs the matrix by index, so a missing or reordered shard is detected, not
  silently mis-joined.
- **Same extraction path as `main.ipynb` cell [6]**: left padding (so `position[-1]` is
  the real last token), `LlamaModel` body only (skip `lm_head`), `use_cache=False`,
  MPS cache emptied per batch. The states are therefore comparable to the old cache.

## Before running
- Time the first shard. The header prints throughput and a full-run ETA; if it is
  unacceptable, stop, lower `CAP` in v14a, rebuild, and start again.
- Disk: roughly `rows x 8 x 4096 x 4 bytes` before compression — about **1.5 GB** for
  ~11,800 rows. Check you have room.
- If `torch` is broken by the macOS dyld change, fix that first — v14c does not need it,
  but this notebook does.

## 1 · Config

In [6]:
import os, json, math, time, gc
import numpy as np
import pandas as pd

MODEL_ID    = "meta-llama/Meta-Llama-3-8B-Instruct"
HOOK_LAYERS = [4, 8, 12, 16, 20, 24, 28, 31]
SYSTEM_PROMPT = ("You are a helpful, secure, and harmless AI assistant. "
                 "You must refuse to assist with any dangerous, illegal, "
                 "or unethical requests.")
MAX_LEN   = 512      # 24 GB machine: keep activations small. v14a's carrier
                     # truncation means BIPIA payloads still survive this cap.
# Dynamic batching. Padding is per-batch to the LONGEST row in it, and attention is
# O(len^2) -- so a random batch of 4 pads three short rows up to one long one and wastes
# most of the compute. Rows are length-sorted inside each shard and packed to a token
# budget, so every batch is near-uniform and short rows run many-at-a-time.
TOKEN_BUDGET = 1024   # approx tokens per batch (batch_rows x padded_len)
MAX_BATCH    = 4      # hard cap on rows per batch
DEEP_CLEAN_EVERY = 25 # full gc + empty_cache every N batches (MPS fragments badly)
# MEMORY BUDGET (24 GB unified):
#   Llama-3-8B fp16 weights ~16.1 GB. macOS + Jupyter take several more. That leaves
#   only a few GB for activations, and MPS fragments badly across thousands of batches.
#   Symptom of a wedged process: OOM while allocating 256 bytes. No retry can recover
#   that -- restart the kernel.
# Before running: quit browsers and other apps. This job needs the machine near-idle.
SHARD     = 500      # rows per checkpoint file
DTYPE_OUT = np.float32

CORPUS = "datasets/unified/injection_corpus.csv"
OUTDIR = "representation_cache/unified"
os.makedirs(OUTDIR, exist_ok=True)

df = pd.read_csv(CORPUS)
df = df.reset_index(drop=True)
texts = df.text.astype(str).tolist()
N = len(texts)
n_shards = math.ceil(N / SHARD)

done = {int(f.split("_")[1].split(".")[0]) for f in os.listdir(OUTDIR) if f.startswith("shard_")}
todo = [i for i in range(n_shards) if i not in done]

print(f"corpus     : {N:,} rows  ({df.label.sum():,} injections / {(df.label==0).sum():,} benign)")
print(f"shards     : {n_shards} x {SHARD}   done {len(done)}   remaining {len(todo)}")
print(f"est. disk  : {N*len(HOOK_LAYERS)*4096*4/1e9:.2f} GB uncompressed")
print(f"truncation : {MAX_LEN} tokens -- rows longer than this are cut "
      f"({(df.n_chars > MAX_LEN*3).sum():,} rows exceed ~{MAX_LEN*3} chars)")

corpus     : 11,838 rows  (4,148 injections / 7,690 benign)
shards     : 24 x 500   done 24   remaining 0
est. disk  : 1.55 GB uncompressed
truncation : 512 tokens -- rows longer than this are cut (1,058 rows exceed ~1536 chars)


## 2 · Load the model and attach hooks

In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID, token=os.environ.get("HF_TOKEN"))
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"                      # position[-1] is the real last token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16, token=os.environ.get("HF_TOKEN"))
DEV = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(DEV).eval()
base = model.model                             # skip lm_head

BUF = {}
def _mk(li):
    def hook(_m, _i, out):
        h = out[0] if isinstance(out, tuple) else out
        BUF[li] = h[:, -1, :].detach().float().cpu()
    return hook
HANDLES = [base.layers[li].register_forward_hook(_mk(li)) for li in HOOK_LAYERS]

def fmt(p):
    return tok.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": p}],
        tokenize=False, add_generation_prompt=True)

print(f"model on {DEV} | {len(HANDLES)} hooks on layers {HOOK_LAYERS}")
if DEV == "mps":
    print(f"  MPS allocated after load: {torch.mps.current_allocated_memory()/1e9:.2f} GB"
          f" | driver {torch.mps.driver_allocated_memory()/1e9:.2f} GB")
    print("  On 24 GB: expect ~16 GB for weights. If driver is already >20 GB here,"
          " close other apps and restart the kernel before starting §3.")
if DEV != "mps":
    print("\n  *** NOT ON MPS -- extraction will be ~20x slower. Fix torch before continuing. ***")
    print(f"    torch {torch.__version__} | mps built {torch.backends.mps.is_built()} "
          f"| mps available {torch.backends.mps.is_available()}")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

model on mps | 8 hooks on layers [4, 8, 12, 16, 20, 24, 28, 31]
  MPS allocated after load: 16.06 GB | driver 31.51 GB
  On 24 GB: expect ~16 GB for weights. If driver is already >20 GB here, close other apps and restart the kernel before starting §3.


## 3 · Extract

Safe to interrupt. Re-run the cell and it resumes from the next unfinished shard.

In [8]:
def _ntok(t):
    # cheap length proxy; ~4 chars per token for English
    return min(MAX_LEN, max(8, len(t) // 4 + 8))

def _batches(order, texts_local):
    cur, cur_max = [], 0
    for j in order:
        n = _ntok(texts_local[j])
        m = max(cur_max, n)
        if cur and (m * (len(cur) + 1) > TOKEN_BUDGET or len(cur) >= MAX_BATCH):
            yield cur; cur, cur_max = [j], n
        else:
            cur.append(j); cur_max = m
    if cur: yield cur

def run_shard(si):
    lo, hi = si * SHARD, min((si + 1) * SHARD, N)
    idx = np.arange(lo, hi)
    local = texts[lo:hi]
    order = sorted(range(len(local)), key=lambda j: _ntok(local[j]))   # length-sorted
    out = {li: np.empty((len(local), 4096), dtype=DTYPE_OUT) for li in HOOK_LAYERS}
    nb = 0
    def _clean(deep=False):
        if deep: gc.collect()
        if DEV == "mps": torch.mps.empty_cache()

    def _forward(rows_):
        enc = tok([fmt(local[j]) for j in rows_], return_tensors="pt", padding=True,
                  truncation=True, max_length=MAX_LEN).to(DEV)
        with torch.inference_mode():
            base(**enc, output_attentions=False, use_cache=False)
        for li in HOOK_LAYERS:
            out[li][rows_] = BUF[li].numpy().astype(DTYPE_OUT)   # un-permute: write by row
        del enc
        _clean(deep=(nb % DEEP_CLEAN_EVERY == 0))

    def _run(rows_, depth=0):
        # On MPS OOM, halve the batch and retry. A single row that still OOMs is
        # truncated harder rather than lost -- silent gaps would corrupt v14c.
        nonlocal nb
        try:
            _forward(rows_); nb += 1
        except RuntimeError as e:
            if "out of memory" not in str(e).lower() or depth > 6:
                raise
            gc.collect()
            if DEV == "mps": torch.mps.empty_cache()
            if len(rows_) == 1:
                # One row alone cannot exhaust a healthy process. If this fails after a
                # deep clean, memory is wedged -- stop rather than write a corrupt shard.
                _clean(deep=True); time.sleep(2.0)
                global MAX_LEN
                keep = MAX_LEN
                MAX_LEN = max(128, MAX_LEN // 2)
                print(f"      [oom] single row -> retry at MAX_LEN {keep}->{MAX_LEN}")
                try:
                    _forward(rows_); nb += 1
                except RuntimeError as e2:
                    if "out of memory" in str(e2).lower():
                        raise MemoryError(
                            "MPS is wedged -- a single row OOMed after a deep clean.\n"
                            "  Restart the Jupyter kernel, quit other apps, and re-run.\n"
                            f"  Completed shards are safe; this one ({si}) will redo.") from e2
                    raise
                finally:
                    MAX_LEN = keep
            else:
                h = len(rows_) // 2
                print(f"      [oom] splitting batch {len(rows_)} -> {h}+{len(rows_)-h}")
                _run(rows_[:h], depth+1); _run(rows_[h:], depth+1)

    for b in _batches(order, local):
        _run(b)
    np.savez_compressed(os.path.join(OUTDIR, f"shard_{si:04d}.npz"), row_idx=idx,
                        **{f"layer_{li}": out[li] for li in HOOK_LAYERS})
    gc.collect()
    return hi - lo, nb

t_all = time.time()
for n, si in enumerate(todo, 1):
    t0 = time.time()
    rows, nb = run_shard(si)
    dt = time.time() - t0; rate = rows / dt
    left = sum(min((j + 1) * SHARD, N) - j * SHARD for j in todo[n:])
    print(f"  shard {si:04d}  {rows:>4} rows in {nb:>3} batches  {dt:>6.1f}s  {rate:>5.1f} rows/s"
          f"  | remaining {left:,} ~ {left/max(rate,1e-9)/60:>5.1f} min")
print(f"\ndone in {(time.time()-t_all)/60:.1f} min")



done in 0.0 min


## 4 · Verify and release

In [9]:
shards = sorted(f for f in os.listdir(OUTDIR) if f.startswith("shard_"))
cov = np.concatenate([np.load(os.path.join(OUTDIR, f))["row_idx"] for f in shards]) if shards else np.array([])
missing = sorted(set(range(N)) - set(cov.tolist()))
print(f"shards on disk : {len(shards)}")
print(f"rows covered   : {len(cov):,} / {N:,}")
print(f"duplicates     : {len(cov) - len(set(cov.tolist()))}")
print(f"missing rows   : {len(missing)}" + (f"  e.g. {missing[:8]}" if missing else "  <- complete"))

if not missing:
    d0 = np.load(os.path.join(OUTDIR, shards[0]))
    print(f"shape per layer: {d0['layer_4'].shape[1]}d  dtype {d0['layer_4'].dtype}")
    json.dump(dict(model=MODEL_ID, layers=HOOK_LAYERS, system_prompt=SYSTEM_PROMPT,
                   max_len=MAX_LEN, rows=int(N), shard=SHARD, corpus=CORPUS),
              open(os.path.join(OUTDIR, "manifest.json"), "w"), indent=2)
    print("wrote manifest.json -- v14c can run now")

for h in HANDLES: h.remove()
del model, base
gc.collect()
if DEV == "mps": torch.mps.empty_cache()
print("hooks removed, model released")

shards on disk : 24
rows covered   : 11,838 / 11,838
duplicates     : 0
missing rows   : 0  <- complete
shape per layer: 4096d  dtype float32
wrote manifest.json -- v14c can run now
hooks removed, model released


## Next
`v14c_analysis.ipynb` — payload-grouped CV, leave-one-corpus-out, length-matched controls
and the separability ceiling test. Cache-only, so run it in the clean analysis env, not
the one that needs torch.

If extraction was interrupted and some shards are missing, just re-run §3 — it picks up
where it stopped. Do not delete the folder to "start clean" unless you changed v14a's
`CAP` or `SEED`, in which case the row indices no longer match and **all** shards must go.